In [ ]:
!pip install graphviz -q
!apt-get install graphviz -y

Trabalho utilizando a base de dados Titanic

Algoritmo ID3

In [ ]:
import math
from collections import Counter
import pprint
import csv
import random

# 1. FUNÇÕES DE CARREGAMENTO E VALIDAÇÃO DE DADOS

def load_data_from_csv(filename):
    """Carrega dados de um arquivo CSV."""
    try:
        with open(filename, 'r', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            headers = next(reader)
            data = [row for row in reader]
            print(f"Dados carregados de '{filename}' com sucesso.")
            return headers, data
    except FileNotFoundError:
        print(f"ERRO: O arquivo '{filename}' não foi encontrado.")
        return None, None

def validate_data(data, headers):
    """Valida os dados para o algoritmo ID3 (apenas valores categóricos)."""
    for row_idx, row in enumerate(data):
        if len(row) != len(headers):
            print(f"ERRO DE VALIDAÇÃO: A linha {row_idx + 2} tem {len(row)} colunas.")
            return False
        for col_idx, value in enumerate(row):
            if not value or value.isspace():
                print(f"ERRO: Valor vazio na linha {row_idx + 2}, coluna '{headers[col_idx]}'.")
                return False
            if value.isdigit():
                print(f"ERRO: Valor numérico ('{value}') na linha {row_idx + 2}, coluna '{headers[col_idx]}'. ID3 aceita apenas valores categóricos.")
                return False
    print("Validação concluída. Os dados estão no formato correto.")
    return True

# 2. ALGORITMO ID3

def calculate_entropy(data):
    if not data: return 0
    labels = [row[-1] for row in data]
    label_counts = Counter(labels)
    entropy = 0.0
    total_samples = len(labels)
    for label in label_counts:
        probability = label_counts[label] / total_samples
        entropy -= probability * math.log2(probability)
    return entropy

def split_data(data, attribute_index):
    subsets = {}
    for row in data:
        attribute_value = row[attribute_index]
        if attribute_value not in subsets:
            subsets[attribute_value] = []
        new_row = row[:attribute_index] + row[attribute_index+1:]
        subsets[attribute_value].append(new_row)
    return subsets

def find_best_attribute(data, headers):
    base_entropy = calculate_entropy(data)
    best_info_gain = 0.0
    best_attribute_index = -1
    num_attributes = len(data[0]) - 1
    for i in range(num_attributes):
        unique_values = set(row[i] for row in data)
        subsets = {value: [row for row in data if row[i] == value] for value in unique_values}
        new_entropy = sum((len(subset) / len(data)) * calculate_entropy(subset) for subset in subsets.values())
        info_gain = base_entropy - new_entropy
        if info_gain > best_info_gain:
            best_info_gain = info_gain
            best_attribute_index = i
    return best_attribute_index

def id3(data, headers):
    labels = [row[-1] for row in data]
    if len(set(labels)) == 1:
        return labels[0]
    if len(data[0]) == 1:
        return Counter(labels).most_common(1)[0][0]
    best_attribute_index = find_best_attribute(data, headers)
    if best_attribute_index == -1:
        return Counter(labels).most_common(1)[0][0]
    best_attribute_name = headers[best_attribute_index]
    tree = {best_attribute_name: {}}
    remaining_headers = headers[:best_attribute_index] + headers[best_attribute_index+1:]
    subsets = split_data(data, best_attribute_index)
    for attribute_value, subset in subsets.items():
        if not subset:
             tree[best_attribute_name][attribute_value] = Counter(labels).most_common(1)[0][0]
        else:
             tree[best_attribute_name][attribute_value] = id3(subset, remaining_headers)
    return tree

# 3. FUNÇÕES DE AVALIAÇÃO DO MODELO

def train_test_split(data, test_size=0.2):
    shuffled_data = data[:]
    random.shuffle(shuffled_data)
    split_idx = int(len(shuffled_data) * (1 - test_size))
    return shuffled_data[:split_idx], shuffled_data[split_idx:]

def predict(tree, row, headers):
    if isinstance(tree, str): return tree
    attribute_name = list(tree.keys())[0]
    original_headers = [h for h in headers if h != headers[-1]]
    attribute_index = original_headers.index(attribute_name)
    value = row[attribute_index]

    if value in tree[attribute_name]:
        sub_tree = tree[attribute_name][value]
        remaining_headers = headers[:attribute_index] + headers[attribute_index+1:]
        return predict(sub_tree, row[:attribute_index] + row[attribute_index+1:], remaining_headers)
    else:
        return None

def extract_rules(tree, current_rule="SE"):
    if isinstance(tree, str):
        print(f"{current_rule} ENTÃO classe = {tree}")
        return
    attribute_name = list(tree.keys())[0]
    branches = tree[attribute_name]
    for edge_label, sub_tree in branches.items():
        condition = f"{attribute_name} == '{edge_label}'"
        new_rule = f"{current_rule} {condition}" if current_rule == "SE" else f"{current_rule} E {condition}"
        extract_rules(sub_tree, new_rule)

def calculate_metrics(y_true, y_pred, positive_class='yes'):
    filtered_true = [t for t, p in zip(y_true, y_pred) if p is not None]
    filtered_pred = [p for p in y_pred if p is not None]
    tp = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true == positive_class and pred == positive_class)
    fp = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true != positive_class and pred == positive_class)
    fn = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true == positive_class and pred != positive_class)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1_score

# 4. FUNÇÃO DE VISUALIZAÇÃO

def visualize_tree(tree, dot=None):
    if dot is None:
        from graphviz import Digraph
        dot = Digraph(comment='Árvore de Decisão ID3')
        dot.attr('node', shape='ellipse', style='filled', color='lightblue')
        dot.attr('edge', arrowhead='vee')
    root_id = str(id(tree))
    if isinstance(tree, str):
        dot.node(root_id, label=tree, shape='box', color='lightgreen')
        return dot, root_id
    attribute_name = list(tree.keys())[0]
    dot.node(root_id, label=attribute_name)
    branches = tree[attribute_name]
    for edge_label, sub_tree in branches.items():
        dot, child_id = visualize_tree(sub_tree, dot)
        dot.edge(root_id, child_id, label=str(edge_label))
    return dot, root_id

# 5. EXECUÇÃO PRINCIPAL

if __name__ == '__main__':
    filename = input("Por favor, digite o nome do arquivo CSV para processar (ex: sample_data/tennis_data.csv): ")
    headers, dataset = load_data_from_csv(filename)

    if headers and dataset and validate_data(dataset, headers):
        train_data, test_data = train_test_split(dataset, test_size=0.3)
        print(f"\nDados divididos: {len(train_data)} para treino, {len(test_data)} para teste.")

        print("\nIniciando o treinamento do modelo ID3...")
        decision_tree = id3(train_data, headers[:-1])

        print("\n--- REGRAS EXTRAÍDAS DA ÁRVORE (baseada nos dados de treino) ---")
        extract_rules(decision_tree)
        print("-----------------------------------------------------------------")

        print("\n--- VISUALIZAÇÃO GRÁFICA DA ÁRVORE ---")
        try:
            from IPython.display import display, Image
            dot_graph, _ = visualize_tree(decision_tree)
            output_filename = 'id3_decision_tree'
            dot_graph.render(output_filename, format='png', cleanup=True)
            print(f"Visualização salva como '{output_filename}.png'. Exibindo a imagem abaixo:")
            display(Image(f'{output_filename}.png'))
        except Exception as e:
            print(f"Não foi possível gerar a visualização: {e}")
        print("------------------------------------")

        print("\n--- RESULTADOS DA AVALIAÇÃO (baseada nos dados de teste) ---")
        y_true = [row[-1] for row in test_data]
        y_pred = [predict(decision_tree, row, headers) for row in test_data]
        print(f"Valores Reais:    {y_true}")
        print(f"Valores Previstos: {y_pred}  (None = valor não visto no treino)")

        precision, recall, f1 = calculate_metrics(y_true, y_pred, positive_class='yes')
        print(f"\nMétricas para a classe positiva 'yes':")
        print(f"Precisão: {precision:.2f}")
        print(f"Recall:   {recall:.2f}")
        print(f"F1-Score: {f1:.2f}")
        print("-----------------------------------------------------------------")

Algoritmo C4.5

In [ ]:
import math
from collections import Counter
import csv
import pprint
import random

# 1. FUNÇÕES DE CÁLCULO, VALIDAÇÃO E PRÉ-PROCESSAMENTO

def load_and_prepare_data(filename):
    """
    Carrega dados do CSV, selecionando apenas os atributos relevantes para o Titanic,
    e garante que 'Survived' seja a coluna alvo no final.
    """
    attributes_to_use = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    target_column = 'Survived'

    try:
        with open(filename, 'r', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            full_headers = next(reader)

            all_needed_columns = attributes_to_use + [target_column]
            desired_indices = {col: full_headers.index(col) for col in all_needed_columns}

            final_headers = attributes_to_use + [target_column]

            data = []
            for row in reader:
                if len(row) != len(full_headers): continue
                selected_data = {col_name: row[col_idx] for col_name, col_idx in desired_indices.items()}

                processed_data = {}
                for col_name, value in selected_data.items():
                    if not value.strip():
                        processed_data[col_name] = None
                    elif is_numeric(value):
                        processed_data[col_name] = float(value)
                    else:
                        processed_data[col_name] = value

                ordered_row = [processed_data[h] for h in final_headers]
                data.append(ordered_row)

            print(f"Dados carregados de '{filename}' com sucesso, usando apenas os atributos especificados.")
            return final_headers, data

    except FileNotFoundError:
        print(f"ERRO: O arquivo '{filename}' não foi encontrado.")
        return None, None
    except ValueError as e:
        print(f"ERRO: Uma coluna especificada não foi encontrada no CSV. Detalhe: {e}")
        return None, None

def is_numeric(value):
    try:
        float(value)
        return True
    except (ValueError, TypeError):
        return False

def is_column_numeric(data, column_index):
    for row in data:
        if not isinstance(row[column_index], (int, float)):
            return False
    return True

def handle_missing_data(data, headers):
    if not data: return data
    num_attributes = len(data[0])
    imputation_values = []
    for i in range(num_attributes):
        column_values = [row[i] for row in data if row[i] is not None]
        if not column_values:
            imputation_values.append(None)
            continue
        is_truly_numeric = all(isinstance(v, (int, float)) for v in column_values)
        if is_truly_numeric:
            mean = sum(column_values) / len(column_values)
            imputation_values.append(mean)
            print(f"  - Atributo numérico '{headers[i]}': Média {mean:.2f} será usada.")
        else:
            mode = Counter(column_values).most_common(1)[0][0]
            imputation_values.append(mode)
            print(f"  - Atributo categórico/misto '{headers[i]}': Moda '{mode}' será usada.")
    for row in data:
        for i in range(len(row)):
            if row[i] is None:
                row[i] = imputation_values[i]
    print("Tratamento de dados faltantes concluído.")
    return data

def validate_data(data):
    for row_idx, row in enumerate(data):
        if any(val is None or (isinstance(val, str) and not val.strip()) for val in row):
            print(f"ERRO DE VALIDAÇÃO: Valor vazio na linha {row_idx + 2}.")
            return False
    print("Validação concluída. Os dados parecem corretos.")
    return True

def calculate_entropy(data):
    if not data: return 0
    labels = [row[-1] for row in data]
    label_counts = Counter(labels)
    entropy = 0.0
    total_samples = len(labels)
    for label in label_counts:
        probability = label_counts[label] / total_samples
        if probability > 0:
            entropy -= probability * math.log2(probability)
    return entropy

def calculate_split_info(data, attribute_index):
    total_samples = len(data)
    if total_samples == 0: return 0
    attribute_values = [row[attribute_index] for row in data]
    value_counts = Counter(attribute_values)
    split_info = 0.0
    for value in value_counts:
        probability = value_counts[value] / total_samples
        if probability > 0:
            split_info -= probability * math.log2(probability)
    return split_info

def find_best_numeric_split(data, attribute_index):
    sorted_data = sorted(data, key=lambda x: x[attribute_index])
    best_gain = -1
    best_threshold = -1
    for i in range(len(sorted_data) - 1):
        if sorted_data[i][-1] != sorted_data[i+1][-1]:
            threshold = (sorted_data[i][attribute_index] + sorted_data[i+1][attribute_index]) / 2
            less_than_equal = [row for row in data if row[attribute_index] <= threshold]
            greater_than = [row for row in data if row[attribute_index] > threshold]
            if not less_than_equal or not greater_than: continue
            p_less = len(less_than_equal) / len(data)
            p_greater = len(greater_than) / len(data)
            gain = calculate_entropy(data) - (p_less * calculate_entropy(less_than_equal) + p_greater * calculate_entropy(greater_than))
            if gain > best_gain:
                best_gain = gain
                best_threshold = threshold
    return best_threshold, best_gain

# 2. ALGORITMO C4.5
def find_best_attribute(data, headers, available_indices):
    best_gain_ratio = -1
    best_attribute_index = -1
    best_threshold = None
    base_entropy = calculate_entropy(data)
    if base_entropy == 0: return -1, None
    for i in available_indices:
        is_num = is_column_numeric(data, i)
        if is_num:
            threshold, info_gain = find_best_numeric_split(data, i)
            if info_gain <= 0: continue
            if threshold is None: continue
            p_less = sum(1 for row in data if row[i] <= threshold) / len(data)
            p_greater = 1 - p_less
            split_info = 0
            if p_less > 0: split_info -= p_less * math.log2(p_less)
            if p_greater > 0: split_info -= p_greater * math.log2(p_greater)
        else:
            subsets = {val: [r for r in data if r[i] == val] for val in set(row[i] for row in data)}
            info_gain = base_entropy - sum((len(subset) / len(data)) * calculate_entropy(subset) for subset in subsets.values())
            if info_gain <= 0: continue
            split_info = calculate_split_info(data, i)
        if split_info == 0: continue
        gain_ratio = info_gain / split_info
        if gain_ratio > best_gain_ratio:
            best_gain_ratio = gain_ratio
            best_attribute_index = i
            best_threshold = threshold if is_num else None
    return best_attribute_index, best_threshold

def c45(data, headers, available_indices, min_samples_split=2, max_depth=10, current_depth=0):
    labels = [row[-1] for row in data]
    if len(set(labels)) == 1: return labels[0]
    if not available_indices: return Counter(labels).most_common(1)[0][0]
    if len(data) < min_samples_split: return Counter(labels).most_common(1)[0][0]
    if current_depth >= max_depth: return Counter(labels).most_common(1)[0][0]
    best_attr_idx, threshold = find_best_attribute(data, headers, available_indices)
    if best_attr_idx == -1: return Counter(labels).most_common(1)[0][0]
    best_attr_name = headers[best_attr_idx]
    tree = {best_attr_name: {}}
    is_num = threshold is not None
    if is_num:
        left_data = [row for row in data if row[best_attr_idx] <= threshold]
        right_data = [row for row in data if row[best_attr_idx] > threshold]
        if not left_data or not right_data:
            return Counter(labels).most_common(1)[0][0]
        tree[best_attr_name][f"<= {threshold}"] = c45(left_data, headers, available_indices, min_samples_split, max_depth, current_depth + 1)
        tree[best_attr_name][f"> {threshold}"] = c45(right_data, headers, available_indices, min_samples_split, max_depth, current_depth + 1)
    else:
        next_available_indices = [i for i in available_indices if i != best_attr_idx]
        attr_values = set(row[best_attr_idx] for row in data)
        for value in attr_values:
            subset_data = [row for row in data if row[best_attr_idx] == value]
            if not subset_data:
                tree[best_attr_name][value] = Counter(labels).most_common(1)[0][0]
            else:
                tree[best_attr_name][value] = c45(subset_data, headers, next_available_indices, min_samples_split, max_depth, current_depth + 1)
    return tree

# 3. FUNÇÕES DE AVALIAÇÃO
def train_test_split(data, test_size=0.2, stratify=True):
    if not stratify or len(data) == 0:
        shuffled_data = data[:]
        random.shuffle(shuffled_data)
        split_idx = int(len(shuffled_data) * (1 - test_size))
        return shuffled_data[:split_idx], shuffled_data[split_idx:]
    labels = [row[-1] for row in data]
    data_by_label = {label: [] for label in set(labels)}
    for row in data:
        data_by_label[row[-1]].append(row)
    train_set, test_set = [], []
    for label, rows in data_by_label.items():
        n_test = max(1, int(len(rows) * test_size))
        random.shuffle(rows)
        test_set.extend(rows[:n_test])
        train_set.extend(rows[n_test:])
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set, test_set

def predict(tree, row, headers):
    if not isinstance(tree, dict): return tree
    attribute_name = list(tree.keys())[0]
    try:
        attribute_index = headers.index(attribute_name)
    except ValueError: return None
    value = row[attribute_index]
    branches = tree[attribute_name]
    best_branch = None
    if isinstance(value, (int, float)):
        for edge_label, sub_tree in branches.items():
            if '<=' in edge_label:
                threshold = float(edge_label.split('<=')[1].strip())
                if value <= threshold:
                    best_branch = sub_tree
                    break
            elif '>' in edge_label:
                threshold = float(edge_label.split('>')[1].strip())
                if value > threshold:
                    best_branch = sub_tree
                    break
    elif value in branches:
        best_branch = branches.get(value)
    if best_branch is not None:
        return predict(best_branch, row, headers)
    else: # Se nenhum ramo corresponder
        leaf_predictions = [node for node in branches.values() if not isinstance(node, dict)]
        return Counter(leaf_predictions).most_common(1)[0][0] if leaf_predictions else None


def extract_rules(tree, current_rule="SE"):
    if not isinstance(tree, dict):
        print(f"{current_rule} ENTÃO classe = {tree}")
        return
    attribute_name = list(tree.keys())[0]
    branches = tree[attribute_name]
    for edge_label, sub_tree in branches.items():
        condition = f"{attribute_name} {edge_label}" if any(c in edge_label for c in "<>") else f"{attribute_name} == '{edge_label}'"
        new_rule = f"{current_rule} {condition}" if current_rule == "SE" else f"{current_rule} E {condition}"
        extract_rules(sub_tree, new_rule)

def calculate_metrics(y_true, y_pred, positive_class):
    filtered_true = [t for t, p in zip(y_true, y_pred) if p is not None]
    filtered_pred = [p for p in y_pred if p is not None]
    tp = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true == positive_class and pred == positive_class)
    fp = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true != positive_class and pred == positive_class)
    fn = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true == positive_class and pred != positive_class)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1_score

# 4. FUNÇÃO DE VISUALIZAÇÃO
def visualize_tree(tree, dot=None):
    if dot is None:
        from graphviz import Digraph
        dot = Digraph(comment='Árvore de Decisão C4.5')
        dot.attr('node', shape='ellipse', style='filled', color='skyblue')
        dot.attr('edge', arrowhead='vee')
    root_id = str(id(tree))
    if not isinstance(tree, dict):
        dot.node(root_id, label=str(tree), shape='box', color='lightgreen')
        return dot, root_id
    attribute_name = list(tree.keys())[0]
    dot.node(root_id, label=attribute_name)
    branches = tree[attribute_name]
    for edge_label, sub_tree in branches.items():
        dot, child_id = visualize_tree(sub_tree, dot)
        dot.edge(root_id, child_id, label=str(edge_label))
    return dot, root_id

# 5. EXECUÇÃO PRINCIPAL
if __name__ == '__main__':
    MAX_DEPTH = 5
    MIN_SAMPLES_FOR_SPLIT = 20

    filename = input("Por favor, digite o nome do arquivo CSV para processar (ex: train.csv): ")
    headers, dataset = load_and_prepare_data(filename)

    if headers and dataset:
        print("\n--- TRATAMENTO DE DADOS FALTANTES (IMPUTAÇÃO) ---")
        dataset = handle_missing_data(dataset, headers)
        print("--------------------------------------------------")

        if validate_data(dataset):
            train_data, test_data = train_test_split(dataset, test_size=0.2, stratify=True)
            print(f"\nDados divididos: {len(train_data)} para treino, {len(test_data)} para teste (estratificado).")

            print("\nIniciando o treinamento do modelo C4.5 com pré-poda...")
            initial_indices = list(range(len(headers) - 1))
            decision_tree = c45(train_data, headers, initial_indices,
                                min_samples_split=MIN_SAMPLES_FOR_SPLIT,
                                max_depth=MAX_DEPTH)

            print("\n--- ÁRVORE DE DECISÃO GERADA ---")
            pprint.pprint(decision_tree)
            print("-----------------------------------")

            print("\n--- REGRAS EXTRAÍDAS DA ÁRVORE ---")
            extract_rules(decision_tree)
            print("-------------------------------------------------")

            print("\n--- VISUALIZAÇÃO GRÁFICA DA ÁRVORE ---")
            try:
                from IPython.display import display, Image
                dot_graph, _ = visualize_tree(decision_tree)
                output_filename = 'c45_decision_tree_final.png'
                dot_graph.render(output_filename, format='png', cleanup=True)
                print(f"Visualização salva como '{output_filename}'.")
                display(Image(f'{output_filename}.png'))
            except Exception as e:
                print(f"Não foi possível gerar a visualização: {e}")
            print("------------------------------------")

            print("\n--- RESULTADOS DA AVALIAÇÃO (baseada nos dados de teste) ---")
            y_true = [row[-1] for row in test_data]
            y_pred = [predict(decision_tree, row, headers) for row in test_data]
            print(f"Valores Reais:   {y_true}")
            print(f"Valores Previstos: {y_pred}")

            positive_class_label = 1.0
            precision, recall, f1 = calculate_metrics(y_true, y_pred, positive_class=positive_class_label)
            print(f"\nMétricas para a classe positiva '{positive_class_label}':")
            print(f"Precisão: {precision:.2f}")
            print(f"Recall:   {recall:.2f}")
            print(f"F1-Score: {f1:.2f}")
            print("-----------------------------------------------------------------")

Algoritmo CART

In [ ]:
import math
from collections import Counter
import csv
import pprint
import random

# 1. FUNÇÕES DE CÁLCULO, VALIDAÇÃO E PRÉ-PROCESSAMENTO

def load_and_prepare_data(filename):
    """
    Carrega dados do CSV, selecionando apenas os atributos relevantes para o Titanic,
    e garante que 'Survived' seja a coluna alvo no final.
    """
    attributes_to_use = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    target_column = 'Survived'

    try:
        with open(filename, 'r', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            full_headers = next(reader)

            all_needed_columns = attributes_to_use + [target_column]
            desired_indices = {col: full_headers.index(col) for col in all_needed_columns}

            final_headers = attributes_to_use + [target_column]

            data = []
            for row in reader:
                if len(row) != len(full_headers): continue
                selected_data = {col_name: row[col_idx] for col_name, col_idx in desired_indices.items()}

                processed_data = {}
                for col_name, value in selected_data.items():
                    if not value.strip():
                        processed_data[col_name] = None
                    elif is_numeric(value):
                        processed_data[col_name] = float(value)
                    else:
                        processed_data[col_name] = value

                ordered_row = [processed_data[h] for h in final_headers]
                data.append(ordered_row)

            print(f"Dados carregados de '{filename}' com sucesso, usando apenas os atributos especificados.")
            return final_headers, data

    except FileNotFoundError:
        print(f"ERRO: O arquivo '{filename}' não foi encontrado.")
        return None, None
    except ValueError as e:
        print(f"ERRO: Uma coluna especificada não foi encontrada no CSV. Detalhe: {e}")
        return None, None

def is_numeric(value):
    try:
        float(value)
        return True
    except (ValueError, TypeError):
        return False

def handle_missing_data(data, headers):
    if not data: return data
    num_attributes = len(data[0])
    imputation_values = []
    for i in range(num_attributes):
        column_values = [row[i] for row in data if row[i] is not None]
        if not column_values:
            imputation_values.append(None)
            continue
        is_truly_numeric = all(isinstance(v, (int, float)) for v in column_values)
        if is_truly_numeric:
            mean = sum(column_values) / len(column_values)
            imputation_values.append(mean)
            print(f"  - Atributo numérico '{headers[i]}': Média {mean:.2f} será usada.")
        else:
            mode = Counter(column_values).most_common(1)[0][0]
            imputation_values.append(mode)
            print(f"  - Atributo categórico/misto '{headers[i]}': Moda '{mode}' será usada.")
    for row in data:
        for i in range(len(row)):
            if row[i] is None:
                row[i] = imputation_values[i]
    print("Tratamento de dados faltantes concluído.")
    return data

def validate_data(data):
    for row_idx, row in enumerate(data):
        if any(val is None or (isinstance(val, str) and not val.strip()) for val in row):
            print(f"ERRO DE VALIDAÇÃO: Valor vazio na linha {row_idx + 2}.")
            return False
    print("Validação concluída. Os dados parecem corretos.")
    return True

def calculate_gini_impurity(data):
    if not data: return 0
    labels = [row[-1] for row in data]
    label_counts = Counter(labels)
    impurity = 1.0
    total_samples = len(labels)
    if total_samples == 0: return 1.0
    for label in label_counts:
        probability = label_counts[label] / total_samples
        impurity -= probability**2
    return impurity

# 2. DIVISÃO E CONSTRUÇÃO DA ÁRVORE CART

class DecisionNode:
    def __init__(self, question, true_branch, false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

class LeafNode:
    def __init__(self, data):
        self.predictions = Counter(row[-1] for row in data)
        if self.predictions:
            self.result = self.predictions.most_common(1)[0][0]
        else:
            self.result = None

def find_best_split(data, headers):
    best_gain = 0
    best_question = None
    current_impurity = calculate_gini_impurity(data)
    num_attributes = len(data[0]) - 1
    for col in range(num_attributes):
        unique_values = set(row[col] for row in data)
        for val in unique_values:
            question = (headers[col], val)
            true_rows, false_rows = [], []
            is_val_numeric = isinstance(val, (int, float))
            for row in data:
                value = row[col]
                condition_met = (value <= val) if is_val_numeric else (value == val)
                if condition_met: true_rows.append(row)
                else: false_rows.append(row)
            if not true_rows or not false_rows: continue
            p_true = len(true_rows) / len(data)
            gain = current_impurity - p_true * calculate_gini_impurity(true_rows) - (1 - p_true) * calculate_gini_impurity(false_rows)
            if gain > best_gain:
                best_gain = gain
                best_question = question
    return best_gain, best_question

def cart(data, headers, min_samples_split=2, max_depth=10, current_depth=0):
    if len(data) < min_samples_split or current_depth >= max_depth:
        return LeafNode(data)
    gain, question = find_best_split(data, headers)
    if gain == 0:
        return LeafNode(data)
    true_rows, false_rows = [], []
    col_name, value_to_split = question
    col_idx = headers.index(col_name)
    is_val_numeric = isinstance(value_to_split, (int, float))
    for row in data:
        value = row[col_idx]
        condition_met = (value <= value_to_split) if is_val_numeric else (value == value_to_split)
        if condition_met: true_rows.append(row)
        else: false_rows.append(row)
    true_branch = cart(true_rows, headers, min_samples_split, max_depth, current_depth + 1)
    false_branch = cart(false_rows, headers, min_samples_split, max_depth, current_depth + 1)
    return DecisionNode(question, true_branch, false_branch)

# 3. FUNÇÕES DE AVALIAÇÃO
def train_test_split(data, test_size=0.2, stratify=True):
    if not stratify or len(data) == 0:
        shuffled_data = data[:]
        random.shuffle(shuffled_data)
        split_idx = int(len(shuffled_data) * (1 - test_size))
        return shuffled_data[:split_idx], shuffled_data[split_idx:]
    labels = [row[-1] for row in data]
    data_by_label = {label: [] for label in set(labels)}
    for row in data:
        data_by_label[row[-1]].append(row)
    train_set, test_set = [], []
    for label, rows in data_by_label.items():
        n_test = max(1, int(len(rows) * test_size))
        random.shuffle(rows)
        test_set.extend(rows[:n_test])
        train_set.extend(rows[n_test:])
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set, test_set

def predict(node, row, headers):
    if isinstance(node, LeafNode):
        return node.result
    col_name, value_to_split = node.question
    col_idx = headers.index(col_name)
    value = row[col_idx]
    is_val_numeric = isinstance(value_to_split, (int, float))
    condition_met = (value <= value_to_split) if is_val_numeric else (value == value_to_split)
    if condition_met:
        return predict(node.true_branch, row, headers)
    else:
        return predict(node.false_branch, row, headers)

def extract_rules(node, current_rule="SE"):
    if isinstance(node, LeafNode):
        print(f"{current_rule} ENTÃO classe = {node.result}")
        return
    col, val = node.question
    condition_true = f"{col} <= {val}" if isinstance(val, (int, float)) else f"{col} == '{val}'"
    new_rule_true = f"{current_rule} {condition_true}" if current_rule == "SE" else f"{current_rule} E {condition_true}"
    extract_rules(node.true_branch, new_rule_true)
    condition_false = f"{col} > {val}" if isinstance(val, (int, float)) else f"{col} != '{val}'"
    new_rule_false = f"{current_rule} {condition_false}" if current_rule == "SE" else f"{current_rule} E {condition_false}"
    extract_rules(node.false_branch, new_rule_false)

def calculate_metrics(y_true, y_pred, positive_class):
    filtered_true = [t for t, p in zip(y_true, y_pred) if p is not None]
    filtered_pred = [p for p in y_pred if p is not None]
    tp = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true == positive_class and pred == positive_class)
    fp = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true != positive_class and pred == positive_class)
    fn = sum(1 for true, pred in zip(filtered_true, filtered_pred) if true == positive_class and pred != positive_class)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1_score

# 4. FUNÇÃO DE VISUALIZAÇÃO
def visualize_cart_tree(node, dot=None):
    if dot is None:
        from graphviz import Digraph
        dot = Digraph(comment='Árvore de Decisão CART')
        dot.attr('node', shape='ellipse', style='filled', color='orange')
        dot.attr('edge', arrowhead='vee')
    node_id = str(id(node))
    if isinstance(node, LeafNode):
        label = f"Classe: {node.result}\n" + "\n".join([f"{k}: {v}" for k, v in node.predictions.items()])
        dot.node(node_id, label=label, shape='box', color='lightyellow')
        return dot, node_id
    col, val = node.question
    question_str = f"{col} <= {val}" if isinstance(val, (int, float)) else f"{col} == {val}"
    dot.node(node_id, label=question_str)
    dot, true_child_id = visualize_cart_tree(node.true_branch, dot)
    dot.edge(node_id, true_child_id, label="Verdadeiro")
    dot, false_child_id = visualize_cart_tree(node.false_branch, dot)
    dot.edge(node_id, false_child_id, label="Falso")
    return dot, node_id

# 5. EXECUÇÃO PRINCIPAL
if __name__ == '__main__':
    MAX_DEPTH = 5
    MIN_SAMPLES_FOR_SPLIT = 20

    filename = input("Por favor, digite o nome do arquivo CSV para processar (ex: train.csv): ")
    headers, dataset = load_and_prepare_data(filename)

    if headers and dataset:
        print("\n--- TRATAMENTO DE DADOS FALTANTES (IMPUTAÇÃO) ---")
        dataset = handle_missing_data(dataset, headers)
        print("--------------------------------------------------")

        if validate_data(dataset):
            train_data, test_data = train_test_split(dataset, test_size=0.2, stratify=True)
            print(f"\nDados divididos: {len(train_data)} para treino, {len(test_data)} para teste (estratificado).")

            print("\nIniciando o treinamento do modelo CART com pré-poda...")
            decision_tree = cart(train_data, headers,
                                 min_samples_split=MIN_SAMPLES_FOR_SPLIT,
                                 max_depth=MAX_DEPTH)

            print("Treinamento concluído.")

            print("\n--- REGRAS EXTRAÍDAS DA ÁRVORE ---")
            extract_rules(decision_tree)
            print("-------------------------------------------------")

            print("\n--- VISUALIZAÇÃO GRÁFICA DA ÁRVORE ---")
            try:
                from IPython.display import display, Image
                dot_graph, _ = visualize_cart_tree(decision_tree)
                output_filename = 'cart_decision_tree_final.png'
                dot_graph.render(output_filename, format='png', cleanup=True)
                print(f"Visualização salva como '{output_filename}'.")
                display(Image(f'{output_filename}.png'))
            except Exception as e:
                print(f"Não foi possível gerar a visualização: {e}")
            print("------------------------------------")

            print("\n--- RESULTADOS DA AVALIAÇÃO (baseada nos dados de teste) ---")
            y_true = [row[-1] for row in test_data]
            y_pred = [predict(decision_tree, row, headers) for row in test_data]
            print(f"Valores Reais:   {y_true}")
            print(f"Valores Previstos: {y_pred}")

            positive_class_label = 1.0
            precision, recall, f1 = calculate_metrics(y_true, y_pred, positive_class=positive_class_label)
            print(f"\nMétricas para a classe positiva '{positive_class_label}':")
            print(f"Precisão: {precision:.2f}")
            print(f"Recall:   {recall:.2f}")
            print(f"F1-Score: {f1:.2f}")
            print("-----------------------------------------------------------------")